In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

CPI_PATH = Path("data/raw/psa_cpi_by_commodity_2018_2026.csv")
WAGE_PDF_PATH = Path("data/raw/Summary-of-DMWR-per-Wage-Order-by-Region_Non-Agriculture-1989-to-July-9_2026.pdf")

LABEL_COLS = ["Geolocation", "Commodity Description"]

In [2]:
with open(CPI_PATH, encoding="utf-8", newline="") as f:
    for i in range(4):
        print(i, repr(f.readline()[:100]))

0 '"Consumer Price Index for All Income Households by Commodity Group (2018=100): January 2018 - August'
1 '\r\n'
2 '"Geolocation","Commodity Description","2018 Jan","2018 Feb","2018 Mar","2018 Apr","2018 May","2018 J'
3 '"PHILIPPINES","0 - ALL ITEMS",97.2000000000000,97.9000000000000,98.4000000000000,98.8000000000000,99'


In [3]:
plain = pd.read_csv(CPI_PATH)
print("pd.read_csv(path) shape:", plain.shape)
print("Number of columns found:", len(plain.columns))
print("Header it chose:", plain.columns[0][:70], "...")

pd.read_csv(path) shape: (267, 1)
Number of columns found: 1
Header it chose: Consumer Price Index for All Income Households by Commodity Group (201 ...


In [4]:
raw = pd.read_csv(CPI_PATH, skiprows=1)

print("Shape (rows, columns):", raw.shape)
print("First columns:", raw.columns[:4].tolist())
print("Last columns: ", raw.columns[-3:].tolist())
raw.head(3).iloc[:, :6]

Shape (rows, columns): (266, 119)
First columns: ['Geolocation', 'Commodity Description', '2018 Jan', '2018 Feb']
Last columns:  ['2026 Nov', '2026 Dec', '2026 Ave']


,Geolocation,Commodity Description,2018 Jan,2018 Feb,2018 Mar,2018 Apr
0,PHILIPPINES,0 - ALL ITEMS,97.2,97.9,98.4,98.8
1,PHILIPPINES,01 - FOOD AND NON-ALCOHOLIC BEVERAGES,97.1,97.4,97.6,98.0
2,PHILIPPINES,02 - ALCOHOLIC BEVERAGES AND TOBACCO,92.0,96.3,98.1,99.3


In [5]:
print(raw.dtypes.value_counts())

float64    112
str          7
Name: count, dtype: int64


In [6]:
print("Missing values according to pandas:", int(raw.isna().sum().sum()))

text_value_cols = [c for c in raw.select_dtypes(exclude="number").columns if c not in LABEL_COLS]
print("Value columns stored as text:", text_value_cols)
print("Rows containing '..' in each:", {c: int((raw[c] == "..").sum()) for c in text_value_cols})


Missing values according to pandas: 0
Value columns stored as text: ['2026 Sep', '2026 Oct', '2026 Nov', '2026 Dec', '2026 Ave']
Rows containing '..' in each: {'2026 Sep': 266, '2026 Oct': 266, '2026 Nov': 266, '2026 Dec': 266, '2026 Ave': 266}


In [7]:
print("Duplicate rows:", int(raw.duplicated().sum()))
print("Duplicate region + commodity pairs:", int(raw.duplicated(LABEL_COLS).sum()))
print("Distinct locations:", raw["Geolocation"].nunique())
print("Distinct commodity items:", raw["Commodity Description"].nunique())
print()
print("Rows per location:")
print(raw.groupby("Geolocation").size().value_counts())


Duplicate rows: 0
Duplicate region + commodity pairs: 0
Distinct locations: 19
Distinct commodity items: 14

Rows per location:
14    19
Name: count, dtype: int64


In [8]:
print(raw["Geolocation"].unique().tolist())
print()
for item in raw["Commodity Description"].unique():
    print(item)

['PHILIPPINES', '..National Capital Region (NCR)', '..Cordillera Administrative Region (CAR)', '..Region I (Ilocos Region)', '..Region II (Cagayan Valley)', '..Region III (Central Luzon)', '..Region IV-A (CALABARZON)', '..MIMAROPA Region', '..Region V (Bicol Region)', '..Region VI (Western Visayas)', '..Region VII (Central Visayas)', '..Region VIII (Eastern Visayas)', '..Region IX (Zamboanga Peninsula)', '..Region X (Northern Mindanao)', '..Region XI (Davao Region)', '..Region XII (SOCCSKSARGEN)', '..Bangsamoro Autonomous Region in Muslim Mindanao (BARMM)', '..Region XIII (Caraga)', '..Negros Island Region (NIR)']

0 - ALL ITEMS
01 - FOOD AND NON-ALCOHOLIC BEVERAGES
02 - ALCOHOLIC BEVERAGES AND TOBACCO
03 - CLOTHING AND FOOTWEAR
04 - HOUSING, WATER, ELECTRICITY, GAS, AND OTHER FUELS
05 - FURNISHINGS, HOUSEHOLD EQUIPMENT AND ROUTINE HOUSEHOLD MAINTENANCE
06 - HEALTH
07 - TRANSPORT
08 - INFORMATION AND COMMUNICATION
09 - RECREATION, SPORT AND CULTURE
10 - EDUCATION SERVICES
11 - RESTAURA

In [9]:
cpi = pd.read_csv(CPI_PATH, skiprows=1, na_values="..")
value_cols = [c for c in cpi.columns if c not in LABEL_COLS]

print(cpi.dtypes.value_counts())
print()
print("Value columns:", len(value_cols))
print("Missing values now:", int(cpi.isna().sum().sum()))
missing_by_col = cpi.isna().sum()
print(missing_by_col[missing_by_col > 0])

float64    117
str          2
Name: count, dtype: int64

Value columns: 117
Missing values now: 1330
2026 Sep    266
2026 Oct    266
2026 Nov    266
2026 Dec    266
2026 Ave    266
dtype: int64


In [10]:
print("2018 annual average equals 100 in every row:", bool((cpi["2018 Ave"] == 100).all()))
print("Smallest index value:", cpi[value_cols].min().min())
print("Largest index value: ", cpi[value_cols].max().max())

2018 annual average equals 100 in every row: True
Smallest index value: 86.7
Largest index value:  246.1


In [11]:
values = cpi[value_cols].to_numpy(dtype=float)

print("Array shape (location-commodity rows, periods):", values.shape)
print("NaN count:", int(np.isnan(values).sum()))
print("Minimum / maximum:", np.nanmin(values), "/", np.nanmax(values))
print("Columns that are entirely NaN:", int(np.isnan(values).all(axis=0).sum()),
      "(the five unpublished 2026 columns)")

Array shape (location-commodity rows, periods): (266, 117)
NaN count: 1330
Minimum / maximum: 86.7 / 246.1
Columns that are entirely NaN: 5 (the five unpublished 2026 columns)


In [12]:
all_items = cpi[cpi["Commodity Description"].str.startswith("0 -")].set_index("Geolocation")
annual_cols = [c for c in value_cols if c.endswith("Ave")]
all_items.loc["PHILIPPINES", annual_cols].round(1)

2018 Ave    100.0
2019 Ave    102.4
2020 Ave    104.8
2021 Ave    108.9
2022 Ave    115.3
2023 Ave    122.2
2024 Ave    126.1
2025 Ave    128.2
2026 Ave      NaN
Name: PHILIPPINES, dtype: float64

### Variables and the analytical questions

| Variable | Question |
|---|---|
| All-items index by region and month | Q1: year-on-year inflation and real wage purchasing power |
| Same index compared with wage changes between orders | Q2: regions with the widest gap between wage adjustments and inflation |
| Commodity groups 01 to 13 by region and month | Q3: groups that move with negative real-wage growth |
| Region name and year_month | Join keys to the wage data |

Part B: wage PDF sample

In [13]:
import pdfplumber

with pdfplumber.open(WAGE_PDF_PATH) as pdf:
    page_texts = [page.extract_text() or "" for page in pdf.pages]

print("Pages:", len(page_texts))
print()
for line in page_texts[0].splitlines()[:14]:
    print(line)

Pages: 10

DAILY MINIMUM WAGE RATES PER WAGE ORDER BY REGION
NON-AGRICULTURE
1989 - July 9, 2026
AMOUNT
REGION RA/WO DATE ISSUED EFFECTIVITY DATE
BASIC WAGE ALLOWANCE TOTAL
NCR RA 6727 July 1, 1989 89.00 89.00
WO 01 15-Oct-90 Nov. 1, 1990 106.00 106.00
WO 02 20-Dec-90 Jan. 8, 1991 118.00 118.00
WO 03 29-Nov-93 Dec. 16, 1993 135.00 135.00
April 1, 1994 145.00 145.00
WO 04 16-Jan-96 Feb. 2, 1996 161.00 161.00
May 1, 1996 165.00 165.00
WO 05 21-Jan-97 Feb.6, 1997 180.00 180.00


In [14]:
full_text = "\n".join(page_texts)
header = "DAILY MINIMUM WAGE RATES PER WAGE ORDER BY REGION"

print("Region blocks (rows with 'RA 6727'):", full_text.count("RA 6727"))
print("Page headers repeated in the text:", full_text.count(header))
print("Mentions of NIR / Negros Island:", full_text.count("NIR") + full_text.count("Negros Island"))
print()

examples = {
    "date without a clean format": "Jan. 1/2014",
    "effectivity written as text": "6 months after",
    "typo in a rate": "208-.00-250.00",
}
for label, keyword in examples.items():
    hits = [ln for ln in full_text.splitlines() if keyword in ln]
    print(f"{label}: {hits[0] if hits else 'not found'}")

Region blocks (rows with 'RA 6727'): 16
Page headers repeated in the text: 10
Mentions of NIR / Negros Island: 0

date without a clean format: Jan. 1/2014 414.00-451.00 15.00 429.00-466.00
effectivity written as text: 6 months after
typo in a rate: WO 16 22-May-08 July 6, 2008 193.00-235.00 15.00 208-.00-250.00


In [15]:
lines = full_text.splitlines()

#2
for i, line in enumerate(lines):
    if line.startswith(("CMR ", "XII ", "CARAGA ", "XIII***", "ARMM**", "IV-A****")):
        print(i, line)
print()

#3
print([ln for ln in lines if "15-Oct-90" in ln][0])
print([ln for ln in lines if "21-Oct-01" in ln][0])
print()

#7
for keyword in ["Jan.20, 2027", "December 1, 2026", "Sep. 1, 2026"]:
    print(keyword, "->", [ln for ln in lines if keyword in ln][0])


196 IV-A**** WO 09-A 31-Aug-04 Nov. 1, 2004 197.00-255.00 197.00-255.00
532 CMR RA 6727 July 1, 1989 79.00-89.00 79.00-89.00
533 XII WO 01 21-Oct-90 Nov. 11, 1990 85.00-103.00 85.00-103.00
573 CARAGA RA 6727 July 1, 1989 79.00-89.00 79.00-89.00
574 XIII*** WO 01 17-Aug-98 Sept. 5, 1998 142.00-146.00 142.00-146.00
602 ARMM** RA 6727 July 1, 1989 79.00-89.00 79.00-89.00

WO 01 15-Oct-90 Nov. 1, 1990 106.00 106.00
WO 09 21-Oct-01 Nov. 5, 2001 213.00-250.00 15.00 228.00-265.00

Jan.20, 2027 -> Jan.20, 2027 743.00-780.00 743.00-780.00
December 1, 2026 -> December 1, 2026 480.00 480.00
Sep. 1, 2026 -> Sep. 1, 2026 540.00 540.00


### Problems found in the wage PDF

1. The table header repeats on every page and appears inside the text.
2. Region labels appear only once per region block, are split across two lines for some regions (CMR / XII, CARAGA / XIII***), and carry footnote marks (ARMM**, IV-A****, XIII***).
3. Rates are often ranges (500.00-537.00) and many allowance cells are blank.
4. Effectivity dates come in many formats, and some are text instead of dates.
5. At least one rate has a typo (208-.00-250.00).
6. There are no rows for the Negros Island Region, which exists in the CPI data.
7. The file includes scheduled future rates (for example NCR on Jan. 20, 2027, Region V on Dec. 1, 2026, and Region XI on Sep. 1, 2026), which must be excluded when matching wages to months already published in the CPI.

### Parsing plan 

1. Split the text into region blocks at each "RA 6727" row, read the region label from that line and the next, and strip the footnote marks.
2. Remove repeated headers and footnote marks.
3. Parse dates into real dates and flag the ones that fail.
4. Split ranges into low and high values and keep the total.
5. Fill each rate forward by month and join to the CPI on region and year_month.